# EyeScan VisionFM Quality Gate V3

Colab notebook for training the Dell-side `anterior_quality_gate_v3_visionfm_external_linearprobe` candidate using the official VisionFM External Eye backbone.

This notebook:
- mounts Google Drive
- checks the expected Drive layout before training
- supports GPU when available and CPU fallback when not
- saves the trained artifacts back to Drive under `EyeScan_Models/VisionFM_Quality_Gate_V3`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import shutil
import subprocess
import zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATASETS_ROOT = DRIVE_ROOT / 'Datasets'
VFM_ROOT = DATASETS_ROOT / 'VFM Datasets'
WEIGHTS_PATH = VFM_ROOT / 'VFM_External_weights.pth'
DATASET_ROOT = DATASETS_ROOT / 'teyeds_quality_gate_v1'
DATASET_ZIP = DATASETS_ROOT / 'teyeds_quality_gate_v1.zip'
OUTPUT_DIR = DRIVE_ROOT / 'EyeScan_Models' / 'VisionFM_Quality_Gate_V3'
WORKDIR = Path('/content/eyescan_colab')
VENDOR_DIR = WORKDIR / 'VisionFM'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORKDIR.mkdir(parents=True, exist_ok=True)

print('Drive roots:')
for path in [DATASETS_ROOT, VFM_ROOT, OUTPUT_DIR]:
    print(f'  - {path} :: exists={path.exists()}')

if VFM_ROOT.exists():
    print('\nVFM Datasets contents:')
    for item in sorted(VFM_ROOT.iterdir()):
        print('  -', item.name)

if OUTPUT_DIR.parent.exists():
    print('\nEyeScan_Models contents:')
    for item in sorted(OUTPUT_DIR.parent.iterdir()):
        print('  -', item.name)

if not DATASET_ROOT.exists() and DATASET_ZIP.exists():
    print(f'Extracting {DATASET_ZIP} -> {DATASET_ROOT}')
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        archive.extractall(DATASET_ROOT.parent)

required_paths = [WEIGHTS_PATH, DATASET_ROOT]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required Drive assets:\n'
        + '\n'.join(missing)
        + '\n\nExpected layout:\n'
        + '/content/drive/MyDrive/Datasets/VFM Datasets/VFM_External_weights.pth\n'
        + '/content/drive/MyDrive/Datasets/teyeds_quality_gate_v1/good_capture/...\n'
        + '/content/drive/MyDrive/Datasets/teyeds_quality_gate_v1/needs_recapture/...\n'
        + '/content/drive/MyDrive/EyeScan_Models/VisionFM_Quality_Gate_V3/'
    )

print('\nDataset class folders:')
for item in sorted(DATASET_ROOT.iterdir()):
    print('  -', item.name)


In [ ]:
!pip -q install torch torchvision timm pandas scikit-learn einops packaging tqdm

if not VENDOR_DIR.exists():
    !git clone https://github.com/ABILab-CUHK/VisionFM.git "$VENDOR_DIR"
else:
    print(f'VisionFM repo already exists at {VENDOR_DIR}')


In [ ]:
import copy
import io
import math
import random
import sys
import types
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Notebook will continue on CPU.')

VISIONFM_STATS = {
    'External': ((0.4936253, 0.36324808, 0.25956994), (0.32001, 0.27109432, 0.21991591)),
}

LABELS = ['good_capture', 'needs_recapture']
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}

def index_samples(dataset_root: Path):
    samples = []
    for label in LABELS:
        class_dir = dataset_root / label
        for image_path in sorted(class_dir.glob('*')):
            if image_path.is_file() and image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                samples.append({'path': image_path, 'label': label})
    return samples

samples = index_samples(DATASET_ROOT)
targets = [sample['label'] for sample in samples]
train_samples, temp_samples = train_test_split(
    samples,
    test_size=0.30,
    random_state=SEED,
    stratify=targets,
)
temp_targets = [sample['label'] for sample in temp_samples]
val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_targets,
)

for split_name, split_samples in [('train', train_samples), ('val', val_samples), ('test', test_samples)]:
    counts = {label: sum(1 for sample in split_samples if sample['label'] == label) for label in LABELS}
    print(split_name, counts)

mean, std = VISIONFM_STATS['External']
train_transform = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std),
])
eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

class DirectoryDataset(torch.utils.data.Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        entry = self.samples[index]
        with Image.open(entry['path']) as image:
            image = image.convert('RGB')
        image = self.transform(image)
        return image, LABEL_TO_INDEX[entry['label']]

train_dataset = DirectoryDataset(train_samples, train_transform)
val_dataset = DirectoryDataset(val_samples, eval_transform)
test_dataset = DirectoryDataset(test_samples, eval_transform)

batch_size = 16 if DEVICE.type == 'cuda' else 4
num_workers = 2 if DEVICE.type == 'cuda' else 0

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

def install_utils_shim():
    shim = types.ModuleType('utils')
    shim.trunc_normal_ = torch.nn.init.trunc_normal_
    return shim

def import_visionfm_models(vendor_root: Path):
    if str(vendor_root) not in sys.path:
        sys.path.insert(0, str(vendor_root))
    shim = install_utils_shim()
    previous_utils = sys.modules.get('utils')
    previous_models = sys.modules.get('models')
    previous_vt = sys.modules.get('models.vision_transformer')
    sys.modules['utils'] = shim
    try:
        if 'models' in sys.modules:
            del sys.modules['models']
        if 'models.vision_transformer' in sys.modules:
            del sys.modules['models.vision_transformer']
        import importlib
        return importlib.import_module('models')
    finally:
        if previous_utils is None:
            sys.modules.pop('utils', None)
        else:
            sys.modules['utils'] = previous_utils
        if previous_models is not None:
            sys.modules['models'] = previous_models
        if previous_vt is not None:
            sys.modules['models.vision_transformer'] = previous_vt

def resize_pos_embed(posemb, posemb_new, num_tokens=1, gs_new=()):
    ntok_new = posemb_new.shape[1]
    if num_tokens:
        posemb_tok, posemb_grid = posemb[:, :num_tokens], posemb[0, num_tokens:]
        ntok_new -= num_tokens
    else:
        posemb_tok, posemb_grid = posemb[:, :0], posemb[0]
    gs_old = int(math.sqrt(len(posemb_grid)))
    if not len(gs_new):
        gs_new = [int(math.sqrt(ntok_new))] * 2
    posemb_grid = posemb_grid.reshape(1, gs_old, gs_old, -1).permute(0, 3, 1, 2)
    posemb_grid = nn.functional.interpolate(posemb_grid, size=gs_new, mode='bicubic', align_corners=False)
    posemb_grid = posemb_grid.permute(0, 2, 3, 1).reshape(1, gs_new[0] * gs_new[1], -1)
    return torch.cat([posemb_tok, posemb_grid], dim=1)

def load_pretrained_weights(model, pretrained_weights: Path, checkpoint_key=None):
    state_dict = torch.load(pretrained_weights, map_location='cpu', weights_only=False)
    if checkpoint_key and checkpoint_key in state_dict:
        state_dict = state_dict[checkpoint_key]
    state_dict = {k.replace('module.', '').replace('backbone.', ''): v for k, v in state_dict.items()}
    pos_embed = state_dict.get('pos_embed')
    if pos_embed is not None and pos_embed.shape != model.pos_embed.shape:
        state_dict['pos_embed'] = resize_pos_embed(pos_embed, model.pos_embed, getattr(model, 'num_tokens', 1), model.patch_embed.grid_size)
    msg = model.load_state_dict(state_dict, strict=False)
    print('Loaded VisionFM checkpoint with msg:', msg)

def extract_features(encoder, images, n_last_blocks=4, avgpool_patchtokens=0):
    intermediate_output = encoder.get_intermediate_layers(images, n_last_blocks)
    if avgpool_patchtokens == 0:
        output = [tensor[:, 0] for tensor in intermediate_output]
    elif avgpool_patchtokens == 1:
        output = [torch.mean(intermediate_output[-1][:, 1:], dim=1)]
    elif avgpool_patchtokens == 2:
        output = [tensor[:, 0] for tensor in intermediate_output] + [torch.mean(intermediate_output[-1][:, 1:], dim=1)]
    else:
        raise ValueError('Unsupported avgpool_patchtokens')
    return torch.cat(output, dim=-1)

vision_models = import_visionfm_models(VENDOR_DIR)
encoder = vision_models.__dict__['vit_base'](img_size=[224], patch_size=16, num_classes=0, use_mean_pooling=False)
load_pretrained_weights(encoder, WEIGHTS_PATH)
encoder.to(DEVICE)
encoder.eval()
for param in encoder.parameters():
    param.requires_grad = False

head = nn.Sequential(
    nn.Linear(768 * 4, 512),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(512, len(LABELS)),
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-4, weight_decay=0.05)

def run_epoch():
    head.train()
    running_loss = 0.0
    sample_count = 0
    for images, targets in train_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        optimizer.zero_grad()
        with torch.no_grad():
            features = extract_features(encoder, images)
        logits = head(features)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item()) * images.shape[0]
        sample_count += images.shape[0]
    return running_loss / max(sample_count, 1)

def predict(loader):
    head.eval()
    y_true = []
    y_scores = []
    total_loss = 0.0
    sample_count = 0
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)
            features = extract_features(encoder, images)
            logits = head(features)
            loss = criterion(logits, targets)
            probs = torch.softmax(logits, dim=1)
            y_true.append(targets.cpu().numpy())
            y_scores.append(probs.cpu().numpy())
            total_loss += float(loss.item()) * images.shape[0]
            sample_count += images.shape[0]
    return np.concatenate(y_true), np.concatenate(y_scores), total_loss / max(sample_count, 1)

def classification_summary(y_true, y_pred):
    matrix = np.zeros((len(LABELS), len(LABELS)), dtype=int)
    for truth, pred in zip(y_true, y_pred):
        matrix[int(truth), int(pred)] += 1
    per_class = {}
    for index, label in enumerate(LABELS):
        tp = int(matrix[index, index])
        fp = int(matrix[:, index].sum() - tp)
        fn = int(matrix[index, :].sum() - tp)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        per_class[label] = {
            'precision': round(float(precision), 4),
            'recall': round(float(recall), 4),
            'support': int(matrix[index, :].sum()),
        }
    return {
        'accuracy': float(np.mean(y_true == y_pred)),
        'confusion_matrix': matrix.tolist(),
        'per_class': per_class,
    }

def binary_summary(y_true, positive_scores, threshold, positive_label='needs_recapture'):
    positive_index = LABELS.index(positive_label)
    negative_index = 1 - positive_index
    y_pred = np.where(positive_scores >= threshold, positive_index, negative_index)
    summary = classification_summary(y_true, y_pred)
    sensitivity = summary['per_class'][positive_label]['recall']
    negative_label = next(label for label in LABELS if label != positive_label)
    specificity = summary['per_class'][negative_label]['recall']
    summary.update({
        'positive_label': positive_label,
        'negative_label': negative_label,
        'threshold': round(float(threshold), 2),
        'balanced_accuracy': round((sensitivity + specificity) / 2, 4),
    })
    return summary

def select_binary_threshold(y_true, y_scores, positive_label='needs_recapture'):
    positive_index = LABELS.index(positive_label)
    positive_scores = y_scores[:, positive_index]
    best_summary = None
    for threshold in np.linspace(0.05, 0.95, 19):
        summary = binary_summary(y_true, positive_scores, float(threshold), positive_label)
        if best_summary is None:
            best_summary = summary
            continue
        candidate_key = (summary['balanced_accuracy'], summary['per_class'][positive_label]['recall'], summary['accuracy'])
        best_key = (best_summary['balanced_accuracy'], best_summary['per_class'][positive_label]['recall'], best_summary['accuracy'])
        if candidate_key > best_key:
            best_summary = summary
    return best_summary


In [ ]:
RUN_NAME = 'anterior_quality_gate_v3_visionfm_external_linearprobe'
EPOCHS = 6
PATIENCE = 2

history = []
best_val_loss = float('inf')
best_epoch = -1
best_state = None
patience_remaining = PATIENCE

for epoch in range(EPOCHS):
    train_loss = run_epoch()
    y_true_val, y_scores_val, val_loss = predict(val_loader)
    y_pred_val = np.argmax(y_scores_val, axis=1)
    val_accuracy = float(np.mean(y_true_val == y_pred_val))
    history.append({
        'epoch': epoch,
        'train_loss': round(float(train_loss), 6),
        'val_loss': round(float(val_loss), 6),
        'val_accuracy': round(float(val_accuracy), 6),
    })
    print(f'Epoch {epoch + 1}/{EPOCHS} :: train_loss={train_loss:.4f} :: val_loss={val_loss:.4f} :: val_accuracy={val_accuracy:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {
            'encoder_state_dict': copy.deepcopy(encoder.state_dict()),
            'head_state_dict': copy.deepcopy(head.state_dict()),
        }
        torch.save({
            'run_name': RUN_NAME,
            'labels': LABELS,
            'epoch': epoch,
            **best_state,
        }, OUTPUT_DIR / 'best_model.pth')
        patience_remaining = PATIENCE
    else:
        patience_remaining -= 1
        if patience_remaining <= 0:
            print('Early stopping triggered.')
            break

encoder.load_state_dict(best_state['encoder_state_dict'])
head.load_state_dict(best_state['head_state_dict'])

y_true_val, y_scores_val, val_loss = predict(val_loader)
y_true_test, y_scores_test, test_loss = predict(test_loader)
val_summary = classification_summary(y_true_val, np.argmax(y_scores_val, axis=1))
test_summary = classification_summary(y_true_test, np.argmax(y_scores_test, axis=1))
threshold_summary = select_binary_threshold(y_true_val, y_scores_val, positive_label='needs_recapture')
test_threshold_summary = binary_summary(
    y_true_test,
    y_scores_test[:, LABELS.index('needs_recapture')],
    float(threshold_summary['threshold']),
    positive_label='needs_recapture',
)

metrics = {
    'run_name': RUN_NAME,
    'trainer': 'visionfm_colab_linearprobe',
    'device': str(DEVICE),
    'weights_path': str(WEIGHTS_PATH),
    'dataset_root': str(DATASET_ROOT),
    'labels': LABELS,
    'split_counts': {
        'train': {label: sum(1 for sample in train_samples if sample['label'] == label) for label in LABELS},
        'val': {label: sum(1 for sample in val_samples if sample['label'] == label) for label in LABELS},
        'test': {label: sum(1 for sample in test_samples if sample['label'] == label) for label in LABELS},
    },
    'history': history,
    'best_epoch': int(best_epoch),
    'best_val_loss': float(best_val_loss),
    'val_summary': val_summary,
    'test_summary': test_summary,
    'test_loss': float(test_loss),
    'test_accuracy': float(test_summary['accuracy']),
    'binary_decision': {
        'selection_split': 'val',
        'objective': 'balanced_accuracy',
        'selected_threshold': threshold_summary['threshold'],
        'selected_on_val': threshold_summary,
        'test_at_selected_threshold': test_threshold_summary,
    },
}

train_config = {
    'run_name': RUN_NAME,
    'foundation_model_id': 'visionfm_external_eye',
    'visionfm_modality': 'External',
    'weights_path': str(WEIGHTS_PATH),
    'dataset_root': str(DATASET_ROOT),
    'image_size': [224, 224],
    'batch_size': batch_size,
    'epochs': EPOCHS,
    'learning_rate': 1e-4,
    'weight_decay': 0.05,
    'arch': 'vit_base',
    'patch_size': 16,
    'n_last_blocks': 4,
    'avgpool_patchtokens': 0,
    'freeze_backbone': True,
    'head_hidden_dim': 512,
    'dropout': 0.1,
    'seed': SEED,
}

inference_contract = {
    'artifact_type': 'visionfm_pytorch_classifier',
    'backbone_file': str(WEIGHTS_PATH),
    'checkpoint_file': str(OUTPUT_DIR / 'best_model.pth'),
    'color_mode': 'RGB',
    'image_size': [224, 224],
    'normalization': {'mean': list(mean), 'std': list(std)},
    'label_map': {label: index for index, label in enumerate(LABELS)},
    'decision_strategy': {
        'type': 'binary_threshold',
        'positive_label': 'needs_recapture',
        'selected_threshold': threshold_summary['threshold'],
    },
}

torch.save({
    'run_name': RUN_NAME,
    'labels': LABELS,
    'encoder_state_dict': encoder.state_dict(),
    'head_state_dict': head.state_dict(),
}, OUTPUT_DIR / 'final_model.pth')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
(OUTPUT_DIR / 'label_map.json').write_text(json.dumps({label: index for index, label in enumerate(LABELS)}, indent=2) + '\n', encoding='utf-8')
(OUTPUT_DIR / 'train_config.json').write_text(json.dumps(train_config, indent=2) + '\n', encoding='utf-8')
(OUTPUT_DIR / 'inference_contract.json').write_text(json.dumps(inference_contract, indent=2) + '\n', encoding='utf-8')
(OUTPUT_DIR / 'split_manifest.json').write_text(json.dumps({
    'train': [str(sample['path']) for sample in train_samples],
    'val': [str(sample['path']) for sample in val_samples],
    'test': [str(sample['path']) for sample in test_samples],
}, indent=2) + '\n', encoding='utf-8')

zip_path = OUTPUT_DIR.parent / 'VisionFM_Quality_Gate_V3_package.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in OUTPUT_DIR.iterdir():
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

print('\nSaved outputs to:', OUTPUT_DIR)
print('Packaged zip:', zip_path)
print(json.dumps({'test_accuracy': metrics['test_accuracy'], 'selected_threshold': threshold_summary['threshold']}, indent=2))
